In [14]:
import matplotlib
import numpy as np
import pandas as pd
from narwhals import Int64
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plt
from sklearn.base import clone

# 1. Load data

In [15]:
train_df = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index_col=0)

# 2. Preprocess

In [ ]:
# Dropping unnecessary columns for modeling
train_date = train_df["event_dt"]
train_df = train_df.drop(["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI", "year", "Zone", "Latitude", "Longitude", "SubjectID"], axis = 1)
# Unfloating zip code
train_df["Zip"] = train_df["Zip"].astype("Int64")
# Defining categorical columns
cat_cols = ["SubjectRace", "Zip", "District", "StopDescription", "VehicleStyle"]
# Defining discrete/numerical columns
dis_cols = ["SubjectGender", "SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear","SubjectHasPhotoID", "hour", "night", "isDriver", "month"]
# Numerical columns we would like to scale
scale_cols = ["SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "hour", "month"]
# Binary columns
bin_cols = ["SubjectGender", "SubjectHasPhotoID", "night", "isDriver"]

# Converting variables into strings
train_df[cat_cols] = train_df[cat_cols].astype(str)

In [ ]:
target = "search_occurred"
X_train = train_df.drop([target], axis = 1)[cat_cols + dis_cols]
y_train = train_df[target]

# 3. Define classes and functions for k-fold preprocessing

In [ ]:
class CatBinner(BaseEstimator, TransformerMixin):
    def __init__(self, cols, min_freq=0.01, fill_value="Other"):
        self.cols = cols
        self.min_freq = min_freq
        self.fill_value = fill_value

    def fit(self, X, y=None):
        self.keep_labels = {}
        for c in self.cols:
            freq = X[c].value_counts(normalize=True)
            self.keep_labels[c] = freq[freq >= self.min_freq].index.tolist()

        return self

    def transform(self, X):
        X_copy = X.copy()
        for c in self.cols:
            labels = self.keep_labels[c]
            X_copy[c] = np.where(X_copy[c].isin(labels) | X_copy[c].isna(), X_copy[c], self.fill_value)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class RoundImputed(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        rounded = X[self.cols].round()
        self.min_ = rounded.min()
        self.max_ = rounded.max()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.cols] = X_copy[self.cols].round().clip(self.min_, self.max_, axis=1)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class OrdEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, inner):
        self.cat_cols = cat_cols
        self.inner = inner

    def _encode(self, X):
        X_enc = X.copy()
        X_enc[self.cat_cols] = self.encoder_.transform(X[self.cat_cols])
        return X_enc

    def _decode(self, X_imp):
        X_dec = X_imp.copy()
        X_dec[self.cat_cols] = self.encoder_.inverse_transform(X_imp[self.cat_cols])
        return X_dec

    def fit_transform(self, X, y=None):
        self.encoder_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
        self.encoder_.fit(X[self.cat_cols])
        self.inner_ = clone(self.inner)
        return self._decode(self.inner_.fit_transform(self._encode(X), y))

    def fit(self, X, y=None):
        self.fit_transform(X, y)
        return self

    def transform(self, X):
        return self._decode(self.inner_.transform(self._encode(X)))

    def set_output(self, *, transform=None):
        return self

def build_pipeline(model, cat_cols, dis_cols, scale_cols):
    imputer = IterativeImputer(estimator=HistGradientBoostingRegressor(max_iter=50, random_state=10), skip_complete=True, max_iter=5, tol=1e-2, random_state=10)

    impute = Pipeline([
        ("imputer", imputer),
        ("round", RoundImputed(cols=cat_cols + dis_cols)),
    ])

    impute.set_output(transform="pandas")

    encode = ColumnTransformer([("encode", OneHotEncoder(handle_unknown="ignore"), cat_cols), ("scale", StandardScaler(), scale_cols)], remainder="passthrough", verbose_feature_names_out=False)

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("imputer", OrdEncoder(cat_cols, impute)),
        ("encode", encode),
        ("model", model)
    ], memory="cache_dir")

    return pipeline

## 3a) Define pipeline

In [ ]:
# We want to fit lasso logistic regression, and we set max_iter = 1000 to avoid convergence warnings
# Saga solver works for l1_ratio = 1, and it is also faster for large datasets which we have
lr = LogisticRegression(l1_ratio=1, max_iter = 1000, random_state=10, solver="saga")

pipeline = build_pipeline(lr, cat_cols, bin_cols, scale_cols)

# 4. Define grid search parameters and run grid search
## Relevant parameters is C regularization term

In [ ]:
# Inverse regularization term C does not scale linearily, so we want to use the base 10 log
param_grid = {"model__C":np.logspace(-4,4,20)}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

grid_search = GridSearchCV(cv = cv, estimator = pipeline, param_grid=param_grid, scoring=["roc_auc", "neg_brier_score", "average_precision"], refit = "average_precision", verbose=2)

grid_search.fit(X_train, y_train)

In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_)

grid_results.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\lr_hyp_tuning_grid.csv")

In [ ]:
#grid_results = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\lr_hyp_tuning_grid.csv")

In [ ]:
# Implement one-standard-error rule from Elements of Statistical Learning
ind_opt = grid_results["mean_test_average_precision"].idxmax()
mean_opt = grid_results["mean_test_average_precision"].loc[ind_opt]
se_opt = grid_results["std_test_average_precision"].loc[ind_opt]

# Find smallest value of C that is within 1 SD of largest mean
opt_C = min(grid_results["param_model__C"][(mean_opt - se_opt <= grid_results["mean_test_average_precision"]) & (grid_results["mean_test_average_precision"] <= mean_opt + se_opt)])

# Optimal C
print(opt_C)

In [ ]:
plt.plot(grid_results["param_model__C"], grid_results["mean_test_average_precision"])
plt.fill_between(grid_results["param_model__C"], grid_results["mean_test_average_precision"] - grid_results["std_test_average_precision"], grid_results["mean_test_average_precision"] + grid_results["std_test_average_precision"],  alpha=0.2, label="1 SD Bound"
)
plt.axline(xy1 = (opt_C,min(grid_results["mean_test_average_precision"] - grid_results["std_test_average_precision"])), xy2 = (opt_C, max(grid_results["mean_test_average_precision"] + grid_results["std_test_average_precision"])), color = "red", linestyle = "--", label = "Selected C")
plt.legend()
plt.xscale("log")
plt.xlabel("Inverse Regularization Term C")
plt.ylabel("Mean Validation Set PR-AUC/Average Precision")
plt.title("Regularization Term Tuning of Logistic Regression (PR-AUC/Average Precision)")
plt.savefig(r"C:\Users\gelet\PycharmProjects\stop-and-search\plots\logit_hyp_tuning_prauc.png", bbox_inches="tight")
plt.show()

In [ ]:
plt.plot(grid_results["param_model__C"], grid_results["mean_test_roc_auc"])
plt.fill_between(grid_results["param_model__C"], grid_results["mean_test_roc_auc"] - grid_results["std_test_roc_auc"], grid_results["mean_test_roc_auc"] + grid_results["std_test_roc_auc"],  alpha=0.2, label="1 SD Bound"
)
plt.axline(xy1 = (opt_C,min(grid_results["mean_test_roc_auc"] - grid_results["std_test_roc_auc"])), xy2 = (opt_C, max(grid_results["mean_test_roc_auc"] + grid_results["std_test_roc_auc"])), color = "red", linestyle = "--", label = "Selected C")
plt.legend()
plt.xscale("log")
plt.xlabel("Inverse Regularization Term C")
plt.ylabel("Mean Validation Set AUC")
plt.title("Regularization Term Tuning of Logistic Regression (AUC)")
plt.savefig(r"C:\Users\gelet\PycharmProjects\stop-and-search\plots\logit_hyp_tuning_auc.png", bbox_inches="tight")
plt.show()

In [ ]:
plt.plot(grid_results["param_model__C"], -grid_results["mean_test_neg_brier_score"])
plt.fill_between(grid_results["param_model__C"], -grid_results["mean_test_neg_brier_score"] - grid_results["std_test_neg_brier_score"], -grid_results["mean_test_neg_brier_score"] + grid_results["std_test_neg_brier_score"],  alpha=0.2, label="1 SD Bound"
)
plt.axline(xy1 = (opt_C,0.046), xy2 = (opt_C, 0.052), color = "red", linestyle = "--", label = "Selected C")
plt.legend()
plt.xscale("log")
plt.xlabel("Inverse Regularization Term C")
plt.ylabel("Mean Validation Set Brier Score")
plt.title("Regularization Term Tuning of Logistic Regression (Brier)")
plt.savefig(r"C:\Users\gelet\PycharmProjects\stop-and-search\plots\logit_hyp_tuning_brier.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 6), sharex=True, sharey=False)

axs[0].plot(grid_results["param_model__C"], grid_results["mean_test_average_precision"])
axs[0].fill_between(grid_results["param_model__C"], grid_results["mean_test_average_precision"] - grid_results["std_test_average_precision"], grid_results["mean_test_average_precision"] + grid_results["std_test_average_precision"],  alpha=0.2, label="1 SD Bound"
)
axs[0].axline(xy1 = (opt_C,min(grid_results["mean_test_average_precision"] - grid_results["std_test_average_precision"])), xy2 = (opt_C, max(grid_results["mean_test_average_precision"] + grid_results["std_test_average_precision"])), color = "red", linestyle = "--", label = "Selected C")
axs[0].set_ylabel("Mean Validation Set PR-AUC/Average Precision", fontsize = 12)
axs[0].set_title("PR-AUC/Average Precision")

axs[1].plot(grid_results["param_model__C"], grid_results["mean_test_roc_auc"])
axs[1].fill_between(grid_results["param_model__C"], grid_results["mean_test_roc_auc"] - grid_results["std_test_roc_auc"], grid_results["mean_test_roc_auc"] + grid_results["std_test_roc_auc"],  alpha=0.2, label="1 SD Bound"
)
axs[1].axline(xy1 = (opt_C,min(grid_results["mean_test_roc_auc"] - grid_results["std_test_roc_auc"])), xy2 = (opt_C, max(grid_results["mean_test_roc_auc"] + grid_results["std_test_roc_auc"])), color = "red", linestyle = "--", label = "Selected C")
axs[1].set_ylabel("Mean Validation Set AUC", fontsize = 12)
axs[1].set_title("AUC")

axs[2].plot(grid_results["param_model__C"], -grid_results["mean_test_neg_brier_score"])
axs[2].fill_between(grid_results["param_model__C"], -grid_results["mean_test_neg_brier_score"] - grid_results["std_test_neg_brier_score"], -grid_results["mean_test_neg_brier_score"] + grid_results["std_test_neg_brier_score"],  alpha=0.2, label="1 SD Bound"
)
axs[2].axline(xy1 = (opt_C,0.046), xy2 = (opt_C, 0.052), color = "red", linestyle = "--", label = "Selected C")
axs[2].set_ylabel("Mean Validation Set Brier Score", fontsize = 12)
axs[2].set_title("Brier Score")

plt.legend()
plt.xscale("log")
fig.supxlabel("Inverse Regularization Term C")
plt.suptitle("Logistic regression metrics and inverse regularization", size=25)
plt.tight_layout()
plt.savefig(r"C:\Users\gelet\PycharmProjects\stop-and-search\plots\logit_hyp_tuning.png", bbox_inches="tight")
plt.show()

## We use PR-AUC/"Average Precision" as our metric to determine the optimal value of C. We then use the "one-standard-error" rule from Elements of Statistical Learning, which picks the simplest/most parsimonious model that is within one SD of the highest cross-val score. Based on this, the optimal C is 0.012742749857031334.